# Table IV and Appendix Table XIII — zero-shot performance

## Standalone reproduction

This notebook does **not** depend on another TGCM notebook or on previously generated outputs. When opened outside the repository, it automatically downloads the pinned public inference/evaluation support code into `./tgcm_reproduction_support/`.

### Environment

- Python 3.12
- PyTorch 2.6
- NumPy 2.1
- pandas 2.2
- SciPy 1.15
- scikit-learn 1.6
- safetensors 0.5.3
- CUDA 11.8+ is recommended for the full paper run; CPU is supported but slower.

The complete compatible environment is `reproduction/environment.yml`.

### Required evaluation assets

Place these four files either **next to this notebook** or under `tgcm_reproduction_support/data/assets/`:

1. `zero_shot_benchmarks.tar.xz` — ATLAS, NODLINK, ProvCon, DARPA TC-E3 and TC-E5 fixed evaluation records.
2. `checkpoints_tgcm.tar.xz.part-000`
3. `checkpoints_tgcm.tar.xz.part-001` — together, the sanitized TGCM inference checkpoints.
4. `checkpoints_neural_baselines.tar.xz` — sanitized DANet/MossFormer2 inference checkpoints; this experiment uses DANet.

No training set, optimizer state, API key, other notebook output, or 822.44 GiB CAPture raw CSV archive is required.

### Reproduction scope

The full run evaluates five seeds of TGCM and DANet on the fixed zero-shot records. Set `FULL_REPRODUCTION=False` only for a code-path smoke test; smoke-test values are not paper results.

In [ ]:
from pathlib import Path
import shutil
import sys
import urllib.request

SUPPORT_REV = "00f786078cf79f01fe2398cc57b7f8e0c45c70fc"
RAW_BASE = "https://raw.githubusercontent.com/Irish-kw/TGCM_Website/" + SUPPORT_REV + "/reproduction"
NOTEBOOK_DIR = Path.cwd().resolve()

def find_existing_root():
    for candidate in (NOTEBOOK_DIR, *NOTEBOOK_DIR.parents):
        if (candidate / "tgcm_review").is_dir() and (candidate / "data").is_dir():
            return candidate
    return None

ROOT = find_existing_root()
if ROOT is None:
    ROOT = NOTEBOOK_DIR / "tgcm_reproduction_support"
    files = ['tgcm_review/__init__.py', 'tgcm_review/assets.py', 'tgcm_review/datasets.py', 'tgcm_review/metrics.py', 'tgcm_review/models.py', 'tgcm_review/inference.py', 'tgcm_review/paper_experiments.py', 'data/manifest.json', 'environment.yml']
    for relative in files:
        destination = ROOT / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        if not destination.exists():
            url = f"{RAW_BASE}/{relative}"
            print(f"Downloading support file: {relative}")
            urllib.request.urlretrieve(url, destination)

asset_dir = ROOT / "data" / "assets"
asset_dir.mkdir(parents=True, exist_ok=True)
required_assets = ['zero_shot_benchmarks.tar.xz', 'checkpoints_tgcm.tar.xz.part-000', 'checkpoints_tgcm.tar.xz.part-001', 'checkpoints_neural_baselines.tar.xz']
for filename in required_assets:
    source = NOTEBOOK_DIR / filename
    target = asset_dir / filename
    if source.is_file() and not target.exists():
        shutil.copy2(source, target)

missing_assets = [name for name in required_assets if not (asset_dir / name).is_file()]
if missing_assets:
    print("\nRequired evaluation asset files are missing:")
    for name in missing_assets:
        print("  -", name)
    print("\nPlace these files either next to this notebook or under:")
    print(" ", asset_dir)
else:
    print("All required evaluation assets are present.")

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

FULL_REPRODUCTION = True
DEVICE = None  # None selects CUDA when available; use "cpu" explicitly for CPU.
print(f"support_root={ROOT}")
print("FULL PAPER RUN" if FULL_REPRODUCTION else "SMOKE TEST — NOT A PAPER RESULT")

In [ ]:
if missing_assets:
    raise FileNotFoundError(
        "Missing required evaluation assets listed above. "
        "Download/place them next to the notebook, then rerun this cell."
    )

from tgcm_review.paper_experiments import run_tables05_13_zero_shot

detail, summary = run_tables05_13_zero_shot(
    steps=100 if FULL_REPRODUCTION else 5,
    device=DEVICE,
    root=ROOT,
)
summary